In [ ]:
# Execução local (a partir da raiz do repositório) — Colab não é necessário.
# No Colab, descomente as duas linhas abaixo para montar o Drive:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Dependências de análise já instaladas via requirements-analysis.txt
# (scikit-posthocs, ucimlrepo, python-pptx). No Colab, descomente:
# !pip install -U scikit-posthocs

In [ ]:
# No Colab, descomente:
# !pip install ucimlrepo

In [ ]:
import os
import sys

# Rodar a partir da raiz do repositório para que conf/, models/ e utils/
# resolvam como pacotes (substitui o sys.path do Colab).
sys.path.insert(0, os.getcwd())

import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import utils.plot_dashes as plot_dashes
import scikit_posthocs as sp
import time

from collections import Counter
from ucimlrepo import fetch_ucirepo 
from scipy.stats import friedmanchisquare
from conf.processing import DataProcessing
from utils import genetic_algorithm as gen_alg
from models import adaline, ann_models, k_near, kernel_method, tree_based_ensemble as tree_based

pd.set_option('display.float_format', '{:.6f}'.format)

# Reprodutibilidade: SCN/ELM/Adaline e o GA usam o RNG global (não threadam
# self.current_seed). Fixar a semente aqui torna a corrida inteira determinística
# sem alterar a lógica dos modelos. As 30 execuções do GA continuam gerando
# uma distribuição (sorteios distintos em sequência), apenas reprodutível.
np.random.seed(42)
random.seed(42)

# Artefatos (figuras/tabelas) para preencher os placeholders do deck.
FIG_DIR = 'results/figures'
TAB_DIR = 'results/tables'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

Dataset

In [ ]:

all_results = []
all_runs = []
ga_runs = []
pred_store = {}   # por base: {'target_test', 'GA'} para os gráficos real x previsto

def get_datasets():

    # Arquivos locais do repositório (ordenados para execução determinística)
    file_path = 'data'

    for file_name in sorted(os.listdir(file_path)):

        full_path = os.path.join(file_path, file_name)

        yield {
            'name': file_name,
            'data': pd.read_csv(full_path, sep='\t', decimal='.')
        }

    # Air Quality

    air_quality = fetch_ucirepo(id=360)

    umidade = (
        air_quality.data.features[['RH']]
        .replace(-200, np.nan)
        .dropna()
        .iloc[:3000]
        .reset_index(drop=True)
    )

    yield {
        'name': 'air_quality_rh',
        'data': umidade
    }

    # Appliances Energy

    appliances = fetch_ucirepo(id=374)

    consumo = (
        appliances.data.targets[['Appliances']]
        .iloc[:3000]
    )

    yield {
        'name': 'appliances_energy',
        'data': consumo.reset_index(drop=True)
    }

In [ ]:
for dataset in get_datasets():

    dataset_name = dataset['name']
    dataset_data = dataset['data']

    raw_data = DataProcessing(dataset_data, n_lags=5)
    prep_data = raw_data.prepare_data()

    mlp = ann_models.MLP()
    inicio = time.time()
    results_MLP = mlp.train(prep_data)
    print(f"Dataset: {dataset_name}, MLP: {(time.time()-inicio):.2f} s")
    MLP_forecast_v = np.array(results_MLP['pred_valid'])
    MLP_forecast = np.array(results_MLP['pred_test_denom'])
    
    scn = ann_models.GridSCN()
    inicio = time.time()
    results_SCN = scn.train(prep_data)
    print(f"Dataset: {dataset_name}, SCN: {(time.time()-inicio):.2f} s")
    SCN_forecast_v = np.array(results_SCN['pred_valid'])
    SCN_forecast = np.array(results_SCN['pred_test'])

    elm = ann_models.GridELM()
    inicio = time.time()
    results_ELM = elm.train(prep_data)
    print(f"Dataset: {dataset_name}, ELM: {(time.time()-inicio):.2f} s")
    ELM_forecast_v = np.array(results_ELM['pred_valid'])
    ELM_forecast = np.array(results_ELM['pred_test_denom'])

    svr = kernel_method.SVM()
    inicio = time.time()
    results_SVR = svr.train(prep_data)
    print(f"Dataset: {dataset_name}, SVR: {(time.time()-inicio):.2f} s")
    SVR_forecast_v = np.array(results_SVR['pred_valid'])
    SVR_forecast = np.array(results_SVR['pred_test'])
    
    # rf = tree_based.RF()
    # results_RF = rf.train(prep_data)
    # RF_forecast_v = np.array(results_RF['pred_valid'])
    # RF_forecast = np.array(results_RF['pred_test'])

    # gb = tree_based.GBoosting()
    # results_GB = gb.train(prep_data)
    # GB_forecast_v = np.array(results_GB['pred_valid'])
    # GB_forecast = np.array(results_GB['pred_test'])

    ada = adaline.Adaline()
    inicio = time.time()
    results_ADA = ada.train(prep_data)
    print(f"Dataset: {dataset_name}, ADA: {(time.time()-inicio):.2f} s")
    ADA_forecast_v = np.array(results_ADA['pred_valid'])
    ADA_forecast = np.array(results_ADA['pred_test_denom'])

    KNN = k_near.KNEAR()
    inicio = time.time()
    results_KNN = KNN.train(prep_data)
    print(f"Dataset: {dataset_name}, KNN: {(time.time()-inicio):.2f} s")
    KNN_forecast_v = np.array(results_KNN['pred_valid'])
    KNN_forecast = np.array(results_KNN['pred_test_denom'])

    forecast_valid = {
        "MLP": MLP_forecast_v,
        "ELM": ELM_forecast_v,
        "SCN": SCN_forecast_v,
        "SVR": SVR_forecast_v,
        "ADA": ADA_forecast_v,
        "KNN": KNN_forecast_v
    }

    forecast_test = {
        "MLP": MLP_forecast,
        "ELM": ELM_forecast,
        "SCN": SCN_forecast,
        "SVR": SVR_forecast,
        "ADA": ADA_forecast,
        "KNN": KNN_forecast
    }

    ranking = {
        "MLP": results_MLP['best_error'],
        "ELM": results_ELM['best_error'],
        "SCN": results_SCN['best_error'],
        "SVR": results_SVR['best_error'],
        "KNN": results_KNN['best_error'],
        "ADA": results_ADA['best_error'],
    }

    ranking = sorted(
        ranking.items(),
        key=lambda x: x[1]
    )

    selected_models = [model for model, _ in ranking[:4]]

    P_v = np.column_stack(
        [forecast_valid[m] for m in selected_models]
    )

    P = np.column_stack(
        [forecast_test[m] for m in selected_models]
    )


    for x in range(30):
        
        ga = gen_alg.GeneticAlgorithm()
        
        ga_result = ga.execute(P_v, prep_data['target_valid'])

        y_hat = P @ ga_result['best_ind']
        ga_metrics = plot_dashes.get_metrics_error(prep_data['target_test'], y_hat)

        ga_runs.append({

            'Dataset': dataset_name,
            'Run': x,
            'Selected_Models': selected_models,

            'MSE': ga_metrics['MSE'],
            'RMSE': ga_metrics['RMSE'],
            'MAE': ga_metrics['MAE'],
            'MAPE': ga_metrics['MAPE'],

            'Weights': ga_result['best_ind'],
            'Fitness': ga_result['best_fitness'],
            'Fitness_Curve': ga_result['fitness_curve']

        })

    ga_dataset = [ run for run in ga_runs if run["Dataset"] == dataset_name ]

    best = max( 
        ga_dataset,
        key=lambda r: r["Fitness"]
    )

    y_hat_final = P @ best["Weights"]

    # Guarda real x previsto (ensemble GA) por base para os gráficos do Slide 5.
    pred_store[dataset_name] = {
        'target_test': np.asarray(prep_data['target_test']),
        'GA': np.asarray(y_hat_final),
    }

    weights = np.array([1/4]*4)
    ensemble_mean = P @ weights

    MLP = plot_dashes.get_metrics_error(prep_data['target_test'], MLP_forecast)
    SCN = plot_dashes.get_metrics_error(prep_data['target_test'], SCN_forecast)
    ELM = plot_dashes.get_metrics_error(prep_data['target_test'], ELM_forecast)
    SVR = plot_dashes.get_metrics_error(prep_data['target_test'], SVR_forecast)
    ADA = plot_dashes.get_metrics_error(prep_data['target_test'], ADA_forecast)
    KNN = plot_dashes.get_metrics_error(prep_data['target_test'], KNN_forecast)
    EMEAN = plot_dashes.get_metrics_error(prep_data['target_test'], ensemble_mean)
    GA = plot_dashes.get_metrics_error(prep_data['target_test'], y_hat_final)

    all_results.append({
        'dataset': dataset_name,
        'MLP': MLP,
        'SCN': SCN,
        'ELM': ELM,
        'SVR': SVR,
        'ADA': ADA,
        'KNN': KNN,
        'EMEAN': EMEAN,
        'GA': GA
    })

    all_runs.append({
        'dataset': dataset_name,
        'model': 'MLP',
        'runs': results_MLP['lst_results']
    })

    all_runs.append({
        'dataset': dataset_name,
        'model': 'SCN',
        'runs': results_SCN['lst_results']
    })

    all_runs.append({
        'dataset': dataset_name,
        'model': 'ELM',
        'runs': results_ELM['lst_results']
    })

    all_runs.append({
        'dataset': dataset_name,
        'model': 'SVR',
        'runs': results_SVR['lst_results']
    })

    all_runs.append({
        'dataset': dataset_name,
        'model': 'ADA',
        'runs': results_ADA['lst_results']
    })

    all_runs.append({
        'dataset': dataset_name,
        'model': 'KNN',
        'runs': results_KNN['lst_results']
    })
    

In [ ]:
df_results = pd.DataFrame(all_results)
df_runs = pd.DataFrame(all_runs)

In [ ]:
# 1. Tabela Principal (MSE, RMSE, MAE e MAPE)
rows = []

for _, row in df_results.iterrows():

    dataset = row['dataset']

    for model in ['MLP','SCN','ELM','SVR','ADA','KNN', 'EMEAN', 'GA']:
        metrics = row[model]
        rows.append({

            'Dataset': dataset,
            'Model': model,

            'MSE': metrics['MSE'],
            'RMSE': metrics['RMSE'],
            'MAE': metrics['MAE'],
            'MAPE': metrics['MAPE']

        })

df_metrics = pd.DataFrame(rows)
df_metrics.to_csv(f'{TAB_DIR}/metrics_main.csv', index=False)

In [ ]:
df_metrics

In [ ]:
# 2. Média das Métricas por Modelo
df_metrics.groupby('Model')[['MSE','RMSE','MAE','MAPE']].mean()

In [ ]:
df_metrics.groupby("Model").agg({
    "MSE":["mean","std"],
    "RMSE":["mean","std"],
    "MAE":["mean","std"],
    "MAPE":["mean","std"]
})

In [ ]:
# 3. Ranking dos Modelos
ranking = []

for dataset in df_metrics['Dataset'].unique():

    temp = df_metrics[df_metrics['Dataset']==dataset]
    temp = temp.sort_values('MSE')
    temp['Rank'] = range(1,len(temp)+1)
    ranking.append(temp)

df_rank = pd.concat(ranking)

In [ ]:
df_rank

In [ ]:
df_rank.groupby('Model')['Rank'].mean()

In [ ]:
# 4. Média e Desvio das Execuções - Valores NaN

rows=[]

for _,row in df_runs.iterrows():

    for i,mse in enumerate(row['runs']):

        rows.append({

            'Dataset':row['dataset'],
            'Model':row['model'],
            'Run':i,
            'MSE':mse

        })

df_runs_expand = pd.DataFrame(rows)

df_runs_expand.groupby(
    ['Dataset','Model']
)['MSE'].agg(
    ['mean','std','min','max']
)

In [ ]:
# 5. Organizar o GA

df_ga = pd.DataFrame(ga_runs)

In [ ]:
df_ga.groupby('Dataset')['MSE'].agg(
    ['mean', 'std', 'min', 'max']
)

In [ ]:
# 7. Curva de Convergência

ga_result['fitness_curve']
plt.plot(ga_result['fitness_curve'])
plt.xlabel("Generation")
plt.ylabel("Fitness")
plt.grid()
plt.show()

In [ ]:
datasets = [
    "airlines.txt",
    "coloradoRiver.txt",
    "melbmin.txt",
    "lynx.txt",
    "air_quality_rh",
    "appliances_energy"
]

titles = [
    "Airlines",
    "Colorado River",
    "Melbmin",
    "Lynx",
    "Air Quality",
    "Appliances"
]

fig, axes = plt.subplots(2, 3, figsize=(15,8))

axes = axes.flatten()

for ax, dataset, title in zip(axes, datasets, titles):

    fitness_curves = np.array([
        run["Fitness_Curve"]
        for run in ga_runs
        if run["Dataset"] == dataset
    ])

    mean_curve = fitness_curves.mean(axis=0)

    # Todas as execuções
    for curve in fitness_curves:
        ax.plot(
            curve,
            color='tab:blue',
            alpha=0.25,
            linewidth=1
        )

    # Média
    ax.plot(
        mean_curve,
        color='black',
        linewidth=2.5,
        label='Average'
    )

    ax.set_title(title)
    ax.set_xlabel("Generation")
    ax.set_ylabel("Fitness")
    ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{FIG_DIR}/ga_convergence.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
datasets = [
    "airlines.txt",
    "coloradoRiver.txt",
    "melbmin.txt",
    "lynx.txt",
    "air_quality_rh",
    "appliances_energy"
]

titles = [
    "Airlines",
    "Colorado River",
    "Melbmin",
    "Lynx",
    "Air Quality",
    "Appliances"
]

fig, axes = plt.subplots(2, 3, figsize=(15,8))

axes = axes.flatten()

for ax, dataset, title in zip(axes, datasets, titles):

    dados = df_ga[df_ga["Dataset"] == dataset]["MSE"]

    ax.boxplot(dados)

    ax.set_title(title)

    ax.set_ylabel("MSE")

    ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{FIG_DIR}/ga_boxplots.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Friedman

friedman_df = df_metrics.pivot(
    index='Dataset',
    columns='Model',
    values='MSE'

)

In [ ]:
friedman_df

In [ ]:
friedman_result = friedmanchisquare(

friedman_df['MLP'],
friedman_df['SCN'],
friedman_df['ELM'],
friedman_df['SVR'],
friedman_df['ADA'],
friedman_df['KNN'],
friedman_df['EMEAN'],
friedman_df['GA']
)
friedman_result

In [ ]:
nemenyi = sp.posthoc_nemenyi_friedman(
    friedman_df.values
)
nemenyi.index = friedman_df.columns
nemenyi.columns = friedman_df.columns

In [ ]:
nemenyi

In [ ]:
ranks = friedman_df.rank(axis=1, ascending=True)
avg_ranks = ranks.mean(axis=0)
avg_ranks

In [ ]:
from scikit_posthocs import critical_difference_diagram

plt.figure(figsize=(10, 3))

critical_difference_diagram(
    avg_ranks,
    nemenyi
)

plt.savefig(
    f"{FIG_DIR}/critical_difference.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
avg_rank = (
    df_rank
    .groupby('Model')['Rank']
    .mean()
    .sort_values()
)

# Todos com Cinza
colors = ['lightgray'] * len(avg_rank)

# GA destaque
ga_idx = avg_rank.index.get_loc('GA')
colors[ga_idx] = 'tab:blue'

fig, ax = plt.subplots(figsize=(9,4))

bars = ax.barh(
    avg_rank.index,
    avg_rank.values,
    color=colors,
    edgecolor='black',
    height=0.6
)

ax.invert_yaxis()

ax.set_xlabel("Average Rank (Lower is Better)")
ax.set_title("Average Rank of Forecasting Models Across Datasets")

ax.grid(axis='x', linestyle='--', alpha=0.3)

for bar in bars:
    ax.text(
        bar.get_width()+0.05,
        bar.get_y()+bar.get_height()/2,
        f"{bar.get_width():.2f}",
        va='center'
    )

plt.tight_layout()
plt.savefig(
    f"{FIG_DIR}/avg_rank_bar.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# Quais modelos foram escolhidos em cada BASE
selected = []

for run in ga_runs:

    selected.append({

        "Dataset": run["Dataset"],

        "Selected": ", ".join(run["Selected_Models"])
    })

df_selected = (
    pd.DataFrame(selected)
    .drop_duplicates()
)
df_selected

In [ ]:


counter = Counter()

for run in ga_runs:

    counter.update(run["Selected_Models"])

counter

In [ ]:
# Persistência das tabelas restantes para preencher os placeholders do deck.

# Ranking médio (rank por MSE dentro de cada base, média entre bases) — base do CD.
avg_ranks.rename('avg_rank').sort_values().to_frame().to_csv(f'{TAB_DIR}/avg_rank.csv')

# Teste de Friedman: estatística e p-valor.
pd.DataFrame([{
    'statistic': float(friedman_result.statistic),
    'pvalue': float(friedman_result.pvalue),
    'n_datasets': int(friedman_df.shape[0]),
    'k_models': int(friedman_df.shape[1]),
}]).to_csv(f'{TAB_DIR}/friedman.csv', index=False)

# Matriz de Nemenyi (p-valores par-a-par).
nemenyi.to_csv(f'{TAB_DIR}/nemenyi.csv')

# Modelos selecionados (top-4 por erro de validação) por base.
df_selected.to_csv(f'{TAB_DIR}/selected_models.csv', index=False)

# Estatísticas do GA (30 execuções) por base.
df_ga.groupby('Dataset')['MSE'].agg(['mean','std','min','max']).to_csv(f'{TAB_DIR}/ga_mse_stats.csv')

print('Tabelas salvas em', TAB_DIR)
sorted(os.listdir(TAB_DIR))

In [ ]:
# Gráficos real x previsto (ensemble GA) por base — Slide 5.
order = [
    'airlines.txt', 'coloradoRiver.txt', 'melbmin.txt',
    'lynx.txt', 'air_quality_rh', 'appliances_energy'
]
titulos = {
    'airlines.txt': 'Airlines', 'coloradoRiver.txt': 'Colorado River',
    'melbmin.txt': 'Melbmin', 'lynx.txt': 'Lynx',
    'air_quality_rh': 'Air Quality (RH)', 'appliances_energy': 'Appliances Energy'
}

# Painel 2x3 (visão geral)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, name in zip(axes.flatten(), order):
    if name not in pred_store:
        ax.set_visible(False)
        continue
    yt = pred_store[name]['target_test']
    yh = pred_store[name]['GA']
    x = np.arange(len(yt))
    ax.plot(x, yt, label='Real', linewidth=1.5)
    ax.plot(x, yh, label='Previsto (GA)', linestyle='--', linewidth=1.5)
    ax.set_title(titulos.get(name, name))
    ax.set_xlabel('Tempo')
    ax.set_ylabel('Valor')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/predictions_grid.png', dpi=300, bbox_inches='tight')
plt.show()

# Individuais por base (reaproveitam o estilo de plot_dashes.get_prediction_plot)
for name in order:
    if name not in pred_store:
        continue
    yt = pred_store[name]['target_test']
    yh = pred_store[name]['GA']
    plt.figure(figsize=(12, 5))
    plt.plot(yt, label='Real', linewidth=2)
    plt.plot(yh, linestyle='--', label='Previsto (GA)', linewidth=2)
    plt.title(titulos.get(name, name))
    plt.xlabel('Tempo')
    plt.ylabel('Valor')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/pred_{name.replace('.txt', '')}.png", dpi=200, bbox_inches='tight')
    plt.close()

print('Figuras salvas em', FIG_DIR)
sorted(os.listdir(FIG_DIR))